# LS-NEQ 13_prereg_evaluation — Präregistrierte Auswertung der vorhandenen Läufe

**Auswertung über die Stufen C, D, G–L** (`lsne.prereg_eval`, siehe `docs/PREREGISTRATION.md` §2–4 und
`docs/STAGES.md`). Dieses Notebook wird von `scripts/build_notebooks.py` erzeugt; Änderungen bitte dort
vornehmen und neu generieren (`python scripts/build_notebooks.py`).

**Zweck.** Die konfirmatorischen Endpunkte aus den Laufverzeichnissen berechnen, die schon auf Drive liegen —
ohne eine GPU-Stufe neu zu rechnen. Bestätigungsmenge: Testfamilien F01–F12 × Formulierungen {A, B}
(Entwicklungsfamilien DEV01–DEV04 bleiben außen vor); Statistik: Mittel der Familienmittel mit
Familien-Cluster-Bootstrap (10 000 Ziehungen, beide Formulierungen einer Familie gemeinsam gezogen);
Benjamini-Hochberg über die Sekundärgruppe 1–6, 8–9 (wenn G0 besteht), 10, 13 und 14 (8, 9, 13, 14 mit dem
`p_boot` ihrer Tabelle; Tabellen ohne `p_boot` bleiben unkorrigiert und höchstens „offen“). Mindesteffekte
entscheidet das Intervall (untere Grenze ≥ Mindesteffekt), nie der Punktschätzer.

**Laufzeit.** Eine **CPU-Laufzeit genügt** (Laufzeit → Laufzeittyp ändern → CPU): es wird kein Modell geladen.
Die Auswertung liest nur; der Checkpoint-Sweep (Notebook 05, Stufe F) wird nie geöffnet, ein noch laufender
Sweep wird also nicht gestört.

**Eingaben.** Die Laufwurzeln `<wurzel>/<stufe>/<modell>@<revision>/` mit `manifest.json` (`status: ok`):
standardmäßig `MyDrive/LSNE/runs` (+ `runs_trace`, `runs_ctl`) und, falls vorhanden, `MyDrive/LSNE_B/runs`
(+ `runs_trace`, `runs_ctl`), die Verknüpfung zum Ordner des zweiten Kontos (siehe nächste Zelle). Liegt eine
Stufe unter mehreren Wurzeln, gilt der neueste Lauf; Stufe L wird in den Stand-in-Lauf und den
Spurenmodell-Lauf getrennt, Stufe D in den Hauptlauf und den Kontrolllauf von Notebook 14 (`runs_ctl`,
`manifest.control = "token_language"`), der den Hauptlauf nie ersetzt.

**Ausgaben.** `prereg_results.csv` (eine Zeile je Endpunkt und Unterzeile: `id, name, estimate, ci_low, ci_high,
p_two_sided, p_one_sided, n_families, n_rows, rule, verdict, bh_q, deviation_note` + `role, stage, direction,
bh_group, bh_q_endpoint, source, min_effect`), `prereg_results.md` (Bericht), `prereg_runs.csv` (gefundene Läufe) und
`prereg_meta.json` unter `<OUT_ROOT>/prereg_eval/<modell>@<revision>/`; die Dateien einer früheren Auswertung dort
werden vorher nach `archive/<zeitstempel>/` verschoben, nie überschrieben. Eine Abbildung unter
`<OUT_ROOT>/figures/13_prereg_evaluation/`; am Ende eine kurze `summary_13_prereg_evaluation_*.json` in `OUT_ROOT`
(Abbildung und Zusammenfassung tragen feste Namen außerhalb des Ausgabeordners und werden überschrieben).

**Ablauf.** Setup → Drive → Wurzeln wählen → ggf. auf Notebook 12 warten → Auswertung → Kontrolllauf, Code der
Auswertung und Archiv → gefundene Läufe → Ergebnistabelle und Bericht → Übersichtsabbildung → Zusammenfassung.

**Präregistrierung.** Primärer Endpunkt (§2): Δλ(K) = λ(K>0, `lang_dom`) − λ(K=0, `lang_dom`) aus Stufe D
(`damage`, Tiefen-Injektion), nur Zelle C1 (die einzige, deren Injektionsposition die präregistrierte ist:
letztes Token von Einheit 9), dieselbe Differenz für `rand` (Mittel über die Zufallsrichtungen) und die
Differenz der Differenzen; gestützt nur, wenn die untere KI-Grenze von Δλ(K=8) den Mindesteffekt 0,05
erreicht (dieselbe Intervallregel wie `damage_boot` in Stufe D), das KI der Differenz der Differenzen positiv
ist und der Anstieg in K belegt ist (Intervall von Δλ beim ersten K und jedes Zuwachses zwischen
aufeinanderfolgenden K über 0; ein Intervall unter 0 widerlegt ihn; PREREGISTRATION 6.10); „offen“, wenn das
Intervall den Mindesteffekt oder den Anstieg weder sichert
noch ausschließt. Ohne eine auf DEV eingefrorene Dosis werden alle präregistrierten Dosen berichtet.
C2/C3/seg erscheinen nur als Abweichungszeilen. §4: Läufe mit `audit_rel_err` > 1e-4 sind ungültig (Zeilen
`audit.D1`, `audit.C`). Kontrast 6: `ent` im Vorzeichen von State 2143 gegen Zufall, einseitig, über alle
Standardvarianten gepoolt; das umgekehrte Vorzeichen ist eine Kontrolle. Kontrast 7 (G0) ist ein Gate: verfehlt
es, werden 8–9 nicht interpretiert. 10 (ii)/(iii) werden auf den Testfamilien neu berechnet (konstante
zensierte Reihen zählen als ρ = 0, die Variante „wie gelaufen“ ist eine Sensitivitätszeile); 10f liest die auf
DEV eingefrorenen Merkmale (`lambda_features_frozen`). 11–18 werden aus ihren Tabellen übernommen (ein
`trace_verdict` nach der alten Regel wird neu beurteilt, nie übernommen); die bekannten Einschränkungen stehen
in `deviation_note`.

**Nachtrag 6.11 (Kontrolle der Sprache an der Injektionsstelle).** In C1 injiziert der Primärendpunkt bei
K = 0 und K > 0 in denselben Punkt „.“ am Ende von Einheit 9; bei K > 0 schließt er einen deutschen Satzteil
ab, und der letzte Wechsel zurück ins Englische folgt direkt danach. Die Zeilen `P.ctl…` (Rolle
`primary_control`, nicht in der BH-Gruppe, m bleibt 26) werten den Kontrolllauf von Notebook 14 mit
derselben Statistik und denselben Regeln wie P aus: Kontrolle 1 (C2 und C3 am Punkt am Ende von Einheit 9)
und Kontrolle 2 (C1 am Punkt am Ende von Einheit 10), je gegen K = 0 an derselben Stelle, Haupt-K = 8;
Kontrolle 3 (nur Einheit 9 deutsch) wird nur berichtet. Ausgang je Dosis in `P.ctl`: „Wechsel“ (beide zeigen
den Effekt), „lokal“ (beide schließen ihn aus) oder „offen“. Die Kontrolle kann P nur schwächen: Bei
„Wechsel“ bleibt P unverändert, bei „lokal“ wird ein gestütztes oder offenes P nicht interpretierbar, bei
„offen“ und **ohne gültigen Kontrolllauf ist P höchstens „offen“**. Das Ergebnis der Regel ohne die Kontrolle
steht in der Notiz von P. Gültig ist der Kontrolllauf nur, wenn seine vier Audits bestehen; sonst sind alle
Zeilen `P.ctl…` nicht interpretierbar:

- `audit.D1ctl`: Schreib-Audit wie §4 (relativer Fehler höchstens 1e-4) über die ganze Tabelle `damage_ctl`;
- `audit.D1ctl.rep`: An der präregistrierten Stelle (`u9`, C1, K = 0, 2, 4, 8) hat jede Kontrollzeile die Zeile
  `depth` des Hauptlaufs mit derselben Variante, Richtung und Dosis an derselben Position, λ weicht höchstens um
  1e-3 ab; umgekehrt hat jede Zeile von P (die Zeilen `depth` des Hauptlaufs bei C1, K = 0, 2, 4, 8 auf der
  Bestätigungsmenge) ihre Kontrollzeile;
- `audit.D1ctl.site`: Jede Zeile liegt in einer Zelle des Designs und injiziert in „.“ mit der richtigen
  Sprachmarke, Einheit und dem richtigen Abstand zum Schlussabschnitt; jede Formulierung der Bestätigungsmenge
  hat jede Zelle des Designs, und jedes Paar aus Variante und Stelle hat genau die Paare aus Richtung und Dosis
  von `u9` / K = 0 / C1, alle mit endlichem λ;
- `audit.D1ctl.run`: Status `ok`, die Marken `control = "token_language"` und `amendment = "6.11"`, jünger als
  der Commit des Nachtrags und kein Rauch- oder eingeschränkter Lauf.

Liegen mehrere fertige Kontrollläufe vor, wird in der Reihenfolge „voller Lauf vor Rauchlauf, ganzes Design vor
eingeschränktem, neuester zuerst“ der erste verwendet, dessen vier Audits bestehen; besteht keiner, wird der
bevorzugte ausgewertet, und es gibt keinen gültigen Kontrolllauf. Die Zelle nach der Auswertung nennt jedes
nicht bestandene Audit in einer Zeile „ACHTUNG: Kontrolllauf ungültig – …“ und, wenn es mehrere Kontrollläufe
gab, die Begründung der Auswahl.

**Herkunft und Archiv.** Der Bericht nennt in der Zeile „Code der Auswertung“ den Git-Stand und den SHA-256 von
`lsne/prereg_eval.py` (in `prereg_meta.json`: `git_sha`, `prereg_eval_sha256`); die Notiz von `audit.D1ctl.run`
vergleicht ihn mit `eval_code_sha256` im Manifest des Kontrolllaufs, denn der Code der Auswertung ist
eingefroren, sobald die Kontrolle gelaufen ist. Ein früherer Bericht im Ausgabeordner (etwa der vom 28.10.2026,
07:07 UTC, auf den sich `docs/RESULTS_PREREG.md` stützt) wird vor dem Schreiben nach `archive/<zeitstempel>/`
verschoben (`prereg_meta.json`: `previous_evaluation_archive`); die Zelle nach der Auswertung nennt den Ordner.

Hinweise: Die Zellen sind gegen Fehler abgesichert (ein fehlgeschlagener Abschnitt druckt den Traceback und das
Notebook läuft weiter). Fehlt eine Stufe, erscheint der Endpunkt mit `not_tested` und dem Grund.


In [ ]:
# --- Setup: Repository holen / aktualisieren und installieren -------------------------
# Runs inside Colab (clone into /content/lsne-repo) and inside an existing checkout
# (local Jupyter, tests): a checkout is detected by lsne/__init__.py + pyproject.toml.
import importlib.util
import os
import pathlib
import subprocess
import sys

GIT_URL = os.environ.get("LSNE_GIT_URL", "https://github.com/Erikiss/Language-Switching-.git")
GIT_BRANCH = os.environ.get("LSNE_GIT_BRANCH", "claude/language-switching-non-equilibrium-ejpz0q")
REPO_DIR = os.environ.get("LSNE_REPO_DIR", "/content/lsne-repo")


def _sh(cmd: str) -> int:
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True).returncode


_here = pathlib.Path.cwd().resolve()
_local = next((p for p in [_here, *_here.parents]
               if (p / "lsne" / "__init__.py").exists() and (p / "pyproject.toml").exists()), None)
if _local is not None:
    REPO_DIR = str(_local)
    print("Vorhandener Checkout:", REPO_DIR)
elif pathlib.Path(REPO_DIR, ".git").exists():
    # a reused runtime: move the shallow checkout to the branch tip (a fast-forward pull fails on
    # depth-1 history and would silently keep the old code)
    if _sh(f"git -C {REPO_DIR} fetch -q --depth 1 origin {GIT_BRANCH} && "
           f"git -C {REPO_DIR} checkout -q -f -B {GIT_BRANCH} FETCH_HEAD") != 0:
        raise RuntimeError(f"{REPO_DIR} konnte nicht auf den Stand von {GIT_BRANCH} gebracht werden: "
                           "Laufzeit -> Sitzung trennen und löschen, dann erneut ausführen")
elif _sh(f"git clone -q --depth 1 --branch {GIT_BRANCH} {GIT_URL} {REPO_DIR}") != 0:
    raise RuntimeError(f"git clone von {GIT_URL} ({GIT_BRANCH}) fehlgeschlagen")
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

if _local is None or importlib.util.find_spec("lsne") is None:
    _sh(f"{sys.executable} -m pip install -q -r requirements.txt")
    _sh(f"{sys.executable} -m pip install -q -e .")
# modules imported by an earlier run in this runtime would shadow the updated checkout
for _m in [m for m in sys.modules if m == "lsne" or m.startswith("lsne.")]:
    del sys.modules[_m]
import lsne
print("lsne", lsne.__version__, "aus", REPO_DIR, "| Branch", GIT_BRANCH)
print("Git-SHA:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())


## Zweites Konto einbinden (`LSNE_B`)

Die Läufe liegen auf zwei Google-Konten. Damit dieses Notebook beide sieht, wird der Ordner des anderen
Kontos als **Verknüpfung** in die eigene Ablage gelegt:

1. Im anderen Konto den Ordner `LSNE` für dieses Konto freigeben (Lesezugriff genügt).
2. In diesem Konto in Google Drive „Für mich freigegeben“ öffnen, Rechtsklick auf `LSNE` →
   „Organisieren“ → „Verknüpfung hinzufügen“ → „Meine Ablage“.
3. Die Verknüpfung in **`LSNE_B`** umbenennen (eine Verknüpfung namens `LSNE` würde mit dem eigenen
   Ordner kollidieren).

Nach dem Einbinden erscheint sie als `/content/drive/MyDrive/LSNE_B`; die Konfigurationszelle nimmt
`LSNE_B/runs` (und `LSNE_B/runs_trace`, `LSNE_B/runs_ctl`) dann automatisch dazu. Andere Wurzeln lassen sich im Feld
`RUN_ROOTS_TEXT` (komma-getrennt) oder mit `ASK = True` in einer Eingabeaufforderung angeben.


In [ ]:
# --- Google Drive einbinden (außerhalb von Colab wird lokal geschrieben) ---------------
# On Colab the results MUST go to Drive: a refused or failed mount stops the notebook
# here (the local disk of the runtime is deleted with the session).  Outside Colab, or
# with LSNE_ALLOW_LOCAL=1, the outputs go to $LSNE_RUNS or ./runs.
import os
import pathlib

try:
    from google.colab import drive            # type: ignore
except ImportError:                           # not on Colab
    drive = None
    print("Kein Colab -> lokale Ausgabe")
if drive is not None and os.environ.get("LSNE_ALLOW_LOCAL") != "1":
    drive.mount("/content/drive")             # confirm the permission dialog in this tab
    if not pathlib.Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Google Drive ist nicht eingebunden: die Ergebnisse würden nur auf der "
                           "Laufzeit liegen und mit ihr gelöscht. Diese Zelle erneut ausführen und den "
                           "Drive-Zugriff im Dialog erlauben.")

from lsne.io import drive_root

OUT_ROOT = drive_root()                       # Colab: /content/drive/MyDrive/LSNE, sonst $LSNE_RUNS oder ./runs
RUNS_ROOT = os.path.join(OUT_ROOT, "runs")    # Laufverzeichnisse wie in jobs/shards.yaml (--out)
os.makedirs(RUNS_ROOT, exist_ok=True)
if OUT_ROOT.startswith("/content/drive"):
    # share the anchor / unigram caches between sessions (see jobs/shards.yaml defaults.env)
    os.environ.setdefault("LSNE_CACHE", os.path.join(OUT_ROOT, "cache"))
print("OUT_ROOT  =", OUT_ROOT)
print("RUNS_ROOT =", RUNS_ROOT)
print("LSNE_CACHE =", os.environ.get("LSNE_CACHE", "./cache"))


In [ ]:
# --- Konfiguration: Laufwurzeln und Auswertungsparameter -------------------------------
NOTEBOOK = "13_prereg_evaluation"
STAGE = "A-L"
MODEL_TAG = "pythia-160m"          # Modell-Tag der Laufverzeichnisse (<stufe>/<modell>@<revision>/)
REVISION = "step98000"             # step98000 (Bestätigungslauf); step99000 für die 99k-Hälfte von Kontrast 11
N_BOOT = 10000                     # präregistriert: 10 000 Familien-Bootstrap-Ziehungen
SEED = 0
FROZEN_DOSE = None                 # auf DEV01-DEV04 eingefrorene Schadensdosis (z. B. 0.05); None = alle Dosen berichten
RUN_ROOTS_TEXT = ""  # @param {type:"string"}
ASK = False                        # True: nach den Wurzeln fragen, wenn RUN_ROOTS_TEXT leer ist (Enter = Standard)
WAIT_FOR_12 = "auto"               # "auto": warten, solange Notebook 12 (Spurenmodell) für REVISION läuft; True: immer; False: nie
WAIT_MAX_H = 10                    # längstens so viele Stunden warten, dann trotzdem auswerten
WAIT_POLL_MIN = 5                  # Prüfabstand in Minuten

MYDRIVE = "/content/drive/MyDrive"
default_roots = [os.path.join(MYDRIVE, base, child) for base in ("LSNE", "LSNE_B")
                 for child in ("runs", "runs_trace", "runs_ctl")      # runs_ctl: control run of notebook 14 (6.11)
                 if os.path.isdir(os.path.join(MYDRIVE, base, child))]
if not default_roots:              # outside Colab: the run roots below OUT_ROOT (lsne.io.drive_root()), as on Drive
    default_roots = [os.path.join(OUT_ROOT, child) for child in ("runs", "runs_trace", "runs_ctl")
                     if os.path.isdir(os.path.join(OUT_ROOT, child))] or [RUNS_ROOT]
if ASK and not RUN_ROOTS_TEXT.strip():
    try:
        RUN_ROOTS_TEXT = input(f"Laufwurzeln, komma-getrennt (Enter = {', '.join(default_roots)}): ")
    except Exception:              # no stdin (headless execution): use the defaults
        RUN_ROOTS_TEXT = ""
RUN_ROOTS = [r.strip() for r in RUN_ROOTS_TEXT.split(",") if r.strip()] or default_roots
if os.path.isdir(MYDRIVE) and not os.path.isdir(os.path.join(MYDRIVE, "LSNE_B")):
    print("Hinweis: kein MyDrive/LSNE_B -> die Läufe des zweiten Kontos fehlen (Verknüpfung siehe Markdown oben).")
EVAL_OUT = os.path.join(OUT_ROOT, "prereg_eval", f"{MODEL_TAG}@{REVISION}")
FIG_DIR = os.path.join(OUT_ROOT, "figures", NOTEBOOK)
os.makedirs(FIG_DIR, exist_ok=True)
CLI_COMMAND = ("python -m lsne.prereg_eval " + " ".join(f"--runs {r}" for r in RUN_ROOTS)
               + f" --out {EVAL_OUT} --model {MODEL_TAG} --revision {REVISION} --n-boot {N_BOOT} --seed {SEED}"
               + (f" --frozen-dose {FROZEN_DOSE}" if FROZEN_DOSE is not None else ""))
print("Laufwurzeln:")
for r in RUN_ROOTS:
    print(f"  {r}  ({'vorhanden' if os.path.isdir(r) else 'FEHLT'})")
print("Ausgabe:", EVAL_OUT)
print("CLI-Äquivalent:\n  " + CLI_COMMAND)


In [ ]:
# --- Hilfsfunktionen: Anzeige und abgesicherte Abschnitte --------------------------------
import contextlib
import json
import traceback

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt


def show(df, n: int = 60):
    """display() in Jupyter / Colab, print() elsewhere."""
    if isinstance(df, pd.DataFrame):
        df = df.head(n)
    try:
        display(df)          # noqa: F821 - provided by IPython
    except NameError:
        print(df.to_string() if hasattr(df, "to_string") else df)


@contextlib.contextmanager
def guarded(title: str):
    """Print the traceback of a failing cell section and carry on."""
    print("==", title)
    try:
        yield
    except Exception:
        traceback.print_exc()
        print(f"[{title}] übersprungen (siehe Traceback)")


## Auf Notebook 12 warten (Endpunkt 18T)

Endpunkt 18T liest den Stufe-L-Lauf des Spurenmodells, den Notebook 12 ganz am Ende schreibt. Mit
`WAIT_FOR_12 = "auto"` (Standard) prüft diese Zelle, ob Notebook 12 gerade läuft: Im Ordner
`MyDrive/LSNE/trace_2016` (Schrittmarker, Logs, `serialize_params.json`) gibt es dann Aktivität, die neuer ist
als die letzte `summary_12_trace_source_<modell>_<revision>_*.json`. Läuft 12, wartet die Zelle und prüft alle
`WAIT_POLL_MIN` Minuten. Sie hört auf, sobald eine neue Zusammenfassung von 12 erscheint (12 fertig) oder ein
Schritt von 12 mit `status: error` endet und danach 15 Minuten nichts mehr geschieht (12 abgebrochen);
spätestens nach `WAIT_MAX_H` Stunden wird trotzdem ausgewertet. Läuft 12 nicht, geht es sofort weiter.
So lässt sich dieses Notebook schon starten, während 12 noch rechnet („Alle ausführen“, Laufzeit CPU).
`WAIT_FOR_12 = False` schaltet das Warten ab, `True` wartet auch ohne erkannte Aktivität.


In [ ]:
# --- Auf Notebook 12 warten (nur wenn es gerade läuft) ----------------------------------
import datetime
import glob
import time

TRACE_ROOT_12 = os.path.join(OUT_ROOT, "trace_2016")
SUMMARY_12_GLOB = os.path.join(OUT_ROOT, f"summary_12_trace_source_{MODEL_TAG}_{REVISION}_*.json")


def _mtime(path) -> float:
    try:
        return os.path.getmtime(path)
    except OSError:
        return 0.0


def _when(t: float) -> str:
    return datetime.datetime.fromtimestamp(t, datetime.timezone.utc).strftime("%d.%m. %H:%M UTC") if t else "-"


def trace_state() -> dict:
    """Newest summary of notebook 12 for REVISION, newest activity in its trace folder and its newest
    step marker (Drive is read through the mount; files written by another session can appear late)."""
    summaries = glob.glob(SUMMARY_12_GLOB)
    markers = glob.glob(os.path.join(TRACE_ROOT_12, "steps", "*.json"))
    files = (markers + glob.glob(os.path.join(TRACE_ROOT_12, "logs", "*"))
             + [os.path.join(TRACE_ROOT_12, d, "serialize_params.json") for d in ("itr", "itr_train")])
    last_marker, t_marker = {}, 0.0
    for m in markers:
        t = _mtime(m)
        if t > t_marker:
            try:
                with open(m, encoding="utf-8") as f:
                    last_marker, t_marker = json.load(f), t
            except (OSError, ValueError):
                pass
    return dict(has_summary=bool(summaries), t_summary=max([_mtime(p) for p in summaries] or [0.0]),
                t_activity=max([_mtime(p) for p in files] or [0.0]), marker=last_marker, t_marker=t_marker)


WAIT_INFO = dict(mode=str(WAIT_FOR_12), waited_min=0.0, outcome="not waited")
st0 = trace_state()
running = (st0["t_activity"] > st0["t_summary"] + 60
           and not (st0["marker"].get("status") == "error" and st0["t_marker"] >= st0["t_activity"] - 1))
if WAIT_FOR_12 is True or str(WAIT_FOR_12).lower() == "true":
    wait = True
elif str(WAIT_FOR_12).lower() == "auto":
    wait = st0["has_summary"] and running
else:
    wait = False
print(f"Notebook 12 ({REVISION}): letzte Zusammenfassung {_when(st0['t_summary'])}, letzte Aktivität "
      f"{_when(st0['t_activity'])}, letzter Schritt {st0['marker'].get('step', '-')} "
      f"({st0['marker'].get('status', '-')}) -> {'warten' if wait else 'nicht warten'}")
if wait:
    t_start, deadline = time.time(), time.time() + float(WAIT_MAX_H) * 3600
    WAIT_INFO["outcome"] = "timeout"
    while time.time() < deadline:
        time.sleep(float(WAIT_POLL_MIN) * 60)
        st = trace_state()
        WAIT_INFO["waited_min"] = round((time.time() - t_start) / 60, 1)
        if st["t_summary"] > st0["t_summary"]:
            WAIT_INFO["outcome"] = "notebook 12 finished"
            print(f"Notebook 12 ist fertig (Zusammenfassung {_when(st['t_summary'])}) -> Auswertung startet.")
            break
        if (st["marker"].get("status") == "error" and st["t_marker"] > st0["t_activity"]
                and st["t_marker"] >= st["t_activity"] - 1 and time.time() - st["t_marker"] > 15 * 60):
            WAIT_INFO["outcome"] = "notebook 12 stopped with an error"
            print(f"Notebook 12 hat mit einem Fehler aufgehört (Schritt {st['marker'].get('step')}, "
                  f"{_when(st['t_marker'])}) -> Auswertung startet ohne neuen Spurenmodell-Lauf.")
            break
        print(f"{_when(time.time())}: Notebook 12 läuft noch (letzter Schritt {st['marker'].get('step', '-')}: "
              f"{st['marker'].get('status', '-')}, letzte Aktivität {_when(st['t_activity'])})", flush=True)
    else:
        print(f"Nach {WAIT_MAX_H} h keine neue Zusammenfassung von Notebook 12 -> Auswertung startet trotzdem.")


In [ ]:
# --- Auswertung: alle präregistrierten Endpunkte aus den vorhandenen Tabellen -----------
# CLI-Äquivalent: python -m lsne.prereg_eval --runs <wurzel> [--runs <wurzel2>] --out <dir> --n-boot 10000
from lsne import prereg_eval as PE

res = PE.evaluate(RUN_ROOTS, model=MODEL_TAG, revision=REVISION, n_boot=N_BOOT, seed=SEED, frozen_dose=FROZEN_DOSE)
paths = PE.write_outputs(res, EVAL_OUT)
for k, p in paths.items():
    if p is not None:
        print(f"{k:9s} {p}")
for n in res["meta"]["notes"]:
    print("Hinweis:", n)


In [ ]:
# --- Kontrolllauf (Nachtrag 6.11), Code der Auswertung und Archiv -------------------------
# Every audit of the control run that does not pass (the run is then no valid control: P at most "open"), the
# reason for the choice among several control runs, the version of this evaluation code and the folder the
# previous report was moved to.
with guarded("Kontrolllauf, Code der Auswertung, Archiv"):
    import textwrap

    tab, meta = res["results"], res["meta"]
    AUDIT_DE = {"supported": "bestanden", "not_supported": "nicht bestanden", "not_tested": "nicht getestet"}
    ctl_audits = tab[tab["id"].astype(str).isin(PE.CTL_AUDIT_IDS)]
    CONTROL_VALID = PE.control_valid(ctl_audits.to_dict("records")) if len(ctl_audits) else None
    for _, a in ctl_audits.iterrows():
        if a["verdict"] != "supported":
            note = a["deviation_note"] if isinstance(a["deviation_note"], str) else ""
            print(textwrap.fill(f"ACHTUNG: Kontrolllauf ungültig – {a['id']}: "
                                f"{AUDIT_DE.get(a['verdict'], a['verdict'])} ({a['verdict']}); {note}",
                                width=120, subsequent_indent="    ", break_long_words=False,
                                break_on_hyphens=False))
    if CONTROL_VALID:
        print("Kontrolllauf gültig: alle vier Audits bestanden (audit.D1ctl, .rep, .site, .run; Nachtrag 6.11).")
    elif CONTROL_VALID is False:
        print("Ohne gültigen Kontrolllauf sind alle Zeilen P.ctl… nicht interpretierbar, und der Primärendpunkt ist "
              "nach Nachtrag 6.11 höchstens „offen“.")
    runs_all = res["runs"]
    ctl_runs = runs_all[runs_all["key"] == "D_ctl"]
    sel = [n for n in meta["notes"] if str(n).startswith(("control run ", "choice among the control runs"))]
    if len(ctl_runs) > 1 or sel:
        used_ctl = ctl_runs.loc[ctl_runs["used"].astype(bool), "path"]
        print(f"Mehrere Kontrollläufe gefunden ({len(ctl_runs)}); verwendet: "
              f"{used_ctl.iloc[0] if len(used_ctl) else 'keiner'}.")
        for n in sel:
            print(textwrap.fill(f"Auswahl des Kontrolllaufs: {n}", width=120, subsequent_indent="    ",
                                break_long_words=False, break_on_hyphens=False))
    sha, src = str(meta.get("git_sha") or ""), str(meta.get("prereg_eval_sha256") or "")
    print(f"Code der Auswertung: Git {sha[:7] or 'unbekannt'}, lsne/prereg_eval.py SHA-256 {src[:12] or 'unbekannt'} "
          "(prereg_meta.json: git_sha, prereg_eval_sha256)")
    used_run = ctl_runs[ctl_runs["used"].astype(bool)]
    if len(used_run):
        try:
            with open(os.path.join(used_run["path"].iloc[0], "manifest.json"), encoding="utf-8") as f:
                ec = str(json.load(f).get("eval_code_sha256") or "")
        except (OSError, ValueError):
            ec = ""
        print(f"Kontrolllauf gerechnet mit lsne/prereg_eval.py SHA-256 {ec[:12] or 'nicht verzeichnet'}: "
              + ("derselbe Code" if ec and ec == src else
                 "ANDERER Code (eine Änderung nach dem Kontrolllauf weicht von Nachtrag 6.11 ab)" if ec else
                 "nicht vergleichbar"))
    arch = paths.get("archive")
    print(f"Frühere Auswertung archiviert nach: {arch}" if arch else
          f"Keine frühere Auswertung in {EVAL_OUT}: nichts archiviert.")


In [ ]:
# --- Gefundene Läufe: welcher Lauf je Stufe verwendet wird ---------------------------
with guarded("Laufverzeichnisse"):
    runs = res["runs"]
    show(runs[["key", "used", "status", "timestamp", "n_boot", "path", "note"]], n=len(runs))
    used = set(runs.loc[runs["used"].astype(bool), "key"])
    missing = [k for k in PE.EVAL_KEYS if k not in used]
    if missing:
        print("Keine verwendbaren Läufe für:", ", ".join(missing), "-> diese Endpunkte erscheinen als not_tested.")
    if "D" in used and "D_ctl" not in used:
        print("Kein Kontrolllauf (Notebook 14, runs_ctl): nach Nachtrag 6.11 ist der Primärendpunkt höchstens „offen“.")


In [ ]:
# --- Ergebnistabelle (alle Endpunkte) und Bericht --------------------------------------
with guarded("Ergebnistabelle"):
    tab = res["results"]
    print("Urteile:", tab["verdict"].value_counts().to_dict())
    print("BH-Gruppe:", res["bh"]["m"], "Tests; Endpunkt-Gruppierung (Sensitivität):", res["bh"]["m_endpoints"], "Endpunkte")
    cols = ["id", "name", "estimate", "ci_low", "ci_high", "p_two_sided", "p_one_sided", "bh_q", "n_families",
            "n_rows", "verdict", "deviation_note"]
    main = tab[tab["role"].isin(["primary", "primary_control", "audit", "secondary", "gate"])]
    show(main[cols], n=len(main))
    print("Weitere Zeilen (Komponenten, Abweichungen, bedingt, übernommen, Sensitivität):",
          int(len(tab) - len(main)), "-> siehe", paths["results"])
with guarded("Bericht (prereg_results.md)"):
    text = paths["markdown"].read_text(encoding="utf-8")
    try:
        from IPython.display import Markdown, display
        display(Markdown(text))
    except ImportError:
        print(text[:6000])


In [ ]:
# --- Übersicht: Schätzer mit 95 %-KI des Primär- und der Sekundärendpunkte ----------------
with guarded("Übersichtsabbildung"):
    tab = res["results"]
    # the control rows P.ctl1-3 per dose (6.11) next to P; their K components stay in the table only
    ctl = tab["role"].eq("primary_control") & tab["id"].astype(str).str.fullmatch(r"P\.ctl[123](@.+)?")
    sel = tab[(tab["role"].isin(["primary", "secondary"]) | ctl) & np.isfinite(tab["ci_low"].astype(float))]
    if sel.empty:
        print("Keine Zeile mit Konfidenzintervall (fehlende Stufen?).")
    else:
        colors = {"supported": "C2", "not_supported": "C3", "open": "C0", "not_interpretable": "grey"}
        fig, ax = plt.subplots(figsize=(8, 0.28 * len(sel) + 1.5))
        y = np.arange(len(sel))[::-1]
        for yi, (_, r) in zip(y, sel.iterrows()):
            c = colors.get(r["verdict"], "k")
            ax.plot([r["ci_low"], r["ci_high"]], [yi, yi], color=c, lw=2)
            ax.plot([r["estimate"]], [yi], "o", color=c, ms=4)
        ax.axvline(0, color="k", lw=0.6)
        ax.set_yticks(y)
        ax.set_yticklabels(sel["id"], fontsize=7)
        ax.set_xlabel("Schätzer [95 %-KI] (Einheit des Endpunkts)")
        ax.set_title("Präregistrierte Endpunkte: grün gestützt, rot nicht gestützt, blau offen", fontsize=9)
        path = os.path.join(FIG_DIR, f"endpoints_{MODEL_TAG}_{REVISION}.png")
        fig.savefig(path, dpi=120, bbox_inches="tight")
        print("Abbildung:", path)
        plt.show()


In [ ]:
# --- Kurze Zusammenfassung nach OUT_ROOT schreiben -------------------------------------
with guarded("Zusammenfassung schreiben"):
    import datetime

    tab = res["results"]
    ctl_audits = tab[tab["id"].astype(str).isin(PE.CTL_AUDIT_IDS)]
    summary = dict(
        notebook=NOTEBOOK, stage=STAGE, model=MODEL_TAG, revision=REVISION, n_boot=N_BOOT, seed=SEED,
        frozen_dose=FROZEN_DOSE, run_roots=RUN_ROOTS, out=EVAL_OUT, cli_command=CLI_COMMAND,
        wait_for_12=WAIT_INFO, timestamp=datetime.datetime.now(datetime.timezone.utc).isoformat(),
        git_sha=res["meta"].get("git_sha"), prereg_eval_sha256=res["meta"].get("prereg_eval_sha256"),
        archive=None if paths.get("archive") is None else str(paths["archive"]),
        runs_used={k: p for k, p, u in zip(res["runs"]["key"], res["runs"]["path"], res["runs"]["used"]) if u},
        verdicts=tab["verdict"].value_counts().to_dict(),
        primary=tab[tab["role"] == "primary"][["id", "estimate", "ci_low", "ci_high", "verdict"]].to_dict("records"),
        primary_control=tab[tab["role"] == "primary_control"][["id", "estimate", "ci_low", "ci_high",
                                                               "verdict"]].to_dict("records"),
        control_valid=PE.control_valid(ctl_audits.to_dict("records")) if len(ctl_audits) else None,
        control_audits=ctl_audits[["id", "verdict"]].to_dict("records"),
        bh=dict(m=res["bh"]["m"], m_endpoints=res["bh"]["m_endpoints"]),
        files={k: str(p) for k, p in paths.items() if p is not None},
        figures=sorted(f for f in os.listdir(FIG_DIR) if f.endswith(".png")),
    )
    path = os.path.join(OUT_ROOT, f"summary_{NOTEBOOK}_{MODEL_TAG}_{REVISION}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=1, ensure_ascii=False, default=str)
    print("Zusammenfassung:", path)
    print(json.dumps(summary, indent=1, ensure_ascii=False, default=str)[:3000])
